In [7]:
import pandas as pd
from pathlib import Path
from python_calamine.pandas import pandas_monkeypatch

files = [
    "上证50.xlsx",
    "中证500.xlsx",
    "中证800指数行情.xlsx",
    "中证1000.xlsx",
    "创业板指数.xlsx",
    "沪深300.xlsx",
]

result = None

pandas_monkeypatch() #补丁用于激活calamine引擎

for file in files:
    df = pd.read_excel(file, engine="calamine")

    df["日期"] = pd.to_datetime(df["日期"]).dt.date
    df = df.sort_values("日期")

    index_name = df["名称"].iloc[0]

    df[f"{index_name}dailyreturn"] = df["收盘价(元)"].pct_change()

    temp = df[["日期", f"{index_name}dailyreturn"]]

    if result is None:
        result = temp
    else:
        result = pd.merge(result, temp, on="日期", how="outer")

result = result.sort_values("日期").reset_index(drop=True)
result = result.fillna(0)
result = result[result['日期']!=0]
result.to_csv(
    "宽基指数收益率.csv",
    index=False,
    encoding="utf-8-sig"
)

print(result.shape)
print(result.head())

(5915, 7)
           日期  上证50dailyreturn  中证500dailyreturn  中证800dailyreturn  \
0  2002-01-04              0.0               0.0               0.0   
1  2002-01-07              0.0               0.0               0.0   
2  2002-01-08              0.0               0.0               0.0   
3  2002-01-09              0.0               0.0               0.0   
4  2002-01-10              0.0               0.0               0.0   

   中证1000dailyreturn  创业板指dailyreturn  沪深300dailyreturn  
0                0.0              0.0          0.000000  
1                0.0              0.0         -0.010916  
2                0.0              0.0         -0.007196  
3                0.0              0.0         -0.015518  
4                0.0              0.0          0.006765  


In [8]:
cols = result.columns.drop('日期')

for col in cols:
    idx = result[col].ne(0).idxmax()

    print(f"{col}:")
    print(f"  第一个非0日期: {result.loc[idx, '日期']}")
    print(f"  值: {result.loc[idx, col]}")

上证50dailyreturn:
  第一个非0日期: 2004-01-02
  值: 0.011349999999999971
中证500dailyreturn:
  第一个非0日期: 2005-01-04
  值: -0.013070000000000026
中证800dailyreturn:
  第一个非0日期: 2005-01-04
  值: -0.016100000000000003
中证1000dailyreturn:
  第一个非0日期: 2005-01-04
  值: -0.010020000000000029
创业板指dailyreturn:
  第一个非0日期: 2010-06-01
  值: -0.02676999999999996
沪深300dailyreturn:
  第一个非0日期: 2002-01-07
  值: -0.010915720308405263


In [10]:
import pandas as pd
df = pd.read_csv("宽基指数收益率.csv")
df.iloc[5914]

日期                   2026-05-27
上证50dailyreturn       -0.015551
中证500dailyreturn      -0.008196
中证800dailyreturn            0.0
中证1000dailyreturn     -0.010663
创业板指dailyreturn         0.00701
沪深300dailyreturn      -0.008088
Name: 5914, dtype: object